In [65]:
BASE_DIR = "/Users/etlevn/Documents/PycharmProjects/LLM_Trd/exp/meta-llama:Meta-Llama-3.1-70B-Instruct"

In [66]:
import os
import json
import pandas as pd
import numpy as np
from scipy.stats import ttest_1samp
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

In [67]:
def analyze_distribution(df):
    """
    Perform error distribution analysis (D.6.1).
    Input:
        df: DataFrame with 'error' column
    Output:
        dict with mean, std, skew, kurtosis
    """
    result = {
        "mean_error": df["error"].mean(),
        "std_error": df["error"].std(),
        "skewness": df["error"].skew(),
        "kurtosis": df["error"].kurtosis()
    }
    return result

In [68]:
def analyze_unbiasedness(df):
    """
    Perform unbiasedness test (D.6.2).
    Input:
        df: DataFrame with 'error' column
    Output:
        dict with t-statistic and p-value
    """
    t_stat, p_val = ttest_1samp(df["error"], popmean=0)
    return {
        "t_stat_error": t_stat,
        "p_val_error": p_val
    }

In [69]:
def analyze_autocorrelation(df):
    """
    Perform autocorrelation test (D.6.3).
    Input:
        df: DataFrame with 'error' and 'agent' columns
    Output:
        dict with AR(1) coefficient, p-value, and Durbin-Watson stat
    """
    # Sort by agent and round to maintain temporal order
    df_sorted = df.sort_values(by=["agent", "predicted_round"]).copy()
    
    # Create lagged error column (within each agent)
    df_sorted["error_lag"] = df_sorted.groupby("agent")["error"].shift(1)
    df_lagged = df_sorted.dropna(subset=["error", "error_lag"])

    # Run OLS regression: error ~ error_lag
    X = sm.add_constant(df_lagged["error_lag"])
    y = df_lagged["error"]
    model = sm.OLS(y, X).fit()

    # Extract AR(1) coefficient, p-value, Durbin-Watson
    ar1_coef = model.params["error_lag"]
    ar1_pval = model.pvalues["error_lag"]
    dw_stat = durbin_watson(model.resid)

    return {
        "AR1_coef": ar1_coef,
        "AR1_pval": ar1_pval,
        "DW_stat": dw_stat
    }

In [70]:
def analyze_llm_experiment(exp_path):
    """
    Run full D.6.1–D.6.3 analysis for one experiment folder.
    Input:
        exp_path: path to one experiment (e.g., /path/to/4o/20250326_...)
    Output:
        dict with all analysis results
    """
    # Load market prices
    mkt_path = os.path.join(exp_path, "market", "mkt.csv")
    prices = []
    try:
        with open(mkt_path, "r") as f:
            for line in f:
                if line.strip() == "" or line.lower().startswith("round"):
                    continue
                _, price, *_ = line.strip().split(",")
                prices.append(float(price))
    except:
        print(f"[warning] Failed to load market prices for {exp_path}")
        return None

    # Collect all forecasts across r1~r20
    forecast_rows = []
    output_dir = os.path.join(exp_path, "output")
    for r in range(1, 21):
        round_dir = os.path.join(output_dir, f"r{r}")
        if not os.path.exists(round_dir):
            continue
        for fname in os.listdir(round_dir):
            if not fname.endswith(".json"):
                continue
            with open(os.path.join(round_dir, fname), "r") as f:
                try:
                    data = json.load(f)
                    forecasts = data.get("new_content", {}).get("price_forecasts", [])
                    agent_id = fname.split("_")[1]
                    for item in forecasts:
                        pred_round = int(item.get("round"))
                        forecast = float(item.get("forecasted_price"))
                        if 1 <= pred_round <= len(prices):
                            true_price = prices[pred_round - 1]
                            error = forecast - true_price
                            forecast_rows.append({
                                "source_round": r,
                                "agent": agent_id,
                                "predicted_round": pred_round,
                                "forecast": forecast,
                                "true_price": true_price,
                                "error": error
                            })
                except:
                    continue

    if len(forecast_rows) == 0:
        print(f"[warning] No valid forecast data found in {exp_path}")
        return None

    df = pd.DataFrame(forecast_rows)

    # Perform all 3 analysis modules
    result = {
        "experiment": os.path.basename(exp_path),
        "MSE": np.mean((df["forecast"] - df["true_price"])**2)
    }
    result.update(analyze_distribution(df))
    result.update(analyze_unbiasedness(df))
    result.update(analyze_autocorrelation(df))
    return result

In [71]:
# Container for results
results = []

# Loop over all subfolders (each is one experiment)
for folder in sorted(os.listdir(BASE_DIR)):
    exp_path = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(exp_path):
        continue  # skip non-folder

    # Skip if this folder doesn't contain a valid market data file
    market_file = os.path.join(exp_path, "market", "mkt.csv")
    if not os.path.exists(market_file):
        print(f"[skip] No market file in: {folder}")
        continue

    # Check if it is a 4-round experiment; if so, skip
    try:
        with open(market_file, "r") as f:
            prices = []
            for line in f:
                if line.strip() == "" or line.lower().startswith("round"):
                    continue
                parts = line.strip().split(",")
                if len(parts) < 2:
                    continue
                price = float(parts[1])
                prices.append(price)

        if len(prices) == 4:
            print(f"[SKIPPED] {folder} has exactly 4 rounds – skipping")
            continue
    except Exception as e:
        print(f"[ERROR] Failed to check round count in {folder}: {e}")
        continue

    print(f"Analyzing: {folder} ...")
    try:
        res = analyze_llm_experiment(exp_path)
        if res is not None:
            results.append(res)
    except Exception as e:
        print(f"[error] {folder} failed: {e}")

# Convert to DataFrame
df_all = pd.DataFrame(results)

# Round all float columns to 4 decimal places
df_all = df_all.round(4)

# Show preview
print(f"\n {len(df_all)} experiments analyzed.")
df_all

[SKIPPED] 20250610_001956 has exactly 4 rounds – skipping
Analyzing: 20250610_010039 ...
[skip] No market file in: plots

 1 experiments analyzed.


,experiment,MSE,mean_error,std_error,skewness,kurtosis,t_stat_error,p_val_error,AR1_coef,AR1_pval,DW_stat
0,20250610_010039,4.6689,0.6214,2.0703,-1.1632,2.73,10.8272,0.0,0.0133,0.634,2.0008


In [72]:
rat_path = os.path.join(BASE_DIR, "rat.csv")
df_all.to_csv(rat_path, index=False)